In [1]:
!pip install -q datasets

In [2]:
from datasets import load_dataset

ds = load_dataset("MedRAG/pubmed", split="train", streaming=True)

sample = list(ds.take(5))

README.md:   0%|          | 0.00/2.56k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/1166 [00:00<?, ?it/s]

In [3]:
for i, row in enumerate(sample):
    print(f"--- Row {i} ---")
    print("ID:", row["id"])
    print("PMID:", row["PMID"])
    print("Title:", row["title"])
    print("Content:", row["content"][:200], "...")   # أول 200 حرف بس
    print()

--- Row 0 ---
ID: pubmed23n0001_0
PMID: 21
Title: [Biochemical studies on camomile components/III. In vitro studies about the antipeptic activity of (--)-alpha-bisabolol (author's transl)].
Content: (--)-alpha-Bisabolol has a primary antipeptic action depending on dosage, which is not caused by an alteration of the pH-value. The proteolytic activity of pepsin is reduced by 50 percent through addi ...

--- Row 1 ---
ID: pubmed23n0001_1
PMID: 22
Title: [Demonstration of tumor inhibiting properties of a strongly immunostimulating low-molecular weight substance. Comparative studies with ifosfamide on the immuno-labile DS carcinosarcoma. Stimulation of the autoimmune activity for approx. 20 days by BA 1, a N-(2-cyanoethylene)-urea. Novel prophylactic possibilities].
Content: A report is given on the recent discovery of outstanding immunological properties in BA 1 [N-(2-cyanoethylene)-urea] having a (low) molecular mass M = 111.104. Experiments in 214 DS carcinosarcoma bea ...

--- Row 2 ---

In [4]:
!pip install -q sentence-transformers

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3", device="cuda")  # هيستخدم الـ GPU بتاعت Colab
print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model loaded. Embedding dimension: 1024


/tmp/ipykernel_469/3376709236.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())


In [6]:
# هنستخدم 'contents' (العنوان + المحتوى مدموجين) مش 'content' لوحده
texts = [row["contents"] for row in sample]

embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print("Shape:", embeddings.shape)   # المفروض يطلع (5, 1024)
print("First vector, first 10 numbers:", embeddings[0][:10])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape: (5, 1024)
First vector, first 10 numbers: [ 0.0242645  -0.01694733 -0.02506855  0.00882224 -0.01816998 -0.08366111
  0.07263766  0.0516577   0.02533822  0.02597569]


In [7]:
!pip install -q qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 10.0 MB/s eta 0:00:00


In [8]:
from getpass import getpass

QDRANT_URL = input("الصقي رابط الـ Cluster هنا: ").strip()
QDRANT_API_KEY = getpass("الصقي الـ API Key هنا (هيبقى مخفي): ").strip()

الصقي رابط الـ Cluster هنا: https://5dc91dc2-c1fb-48c7-9c3a-0260614eaa2c.eu-west-1-0.aws.cloud.qdrant.io
الصقي الـ API Key هنا (هيبقى مخفي): ··········


In [9]:
from qdrant_client import QdrantClient

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

print("Collections currently in cluster:", client.get_collections())

Collections currently in cluster: collections=[]


In [10]:
from qdrant_client.models import Distance, VectorParams

COLLECTION_NAME = "aradoc_pubmed"
VECTOR_SIZE = 1024  # نفس حجم الـ vector اللي طلع من bge-m3

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)

print("Created. Current collections:", client.get_collections())

Created. Current collections: collections=[CollectionDescription(name='aradoc_pubmed')]


In [11]:
from qdrant_client.models import PointStruct

points = []
for i, row in enumerate(sample):
    points.append(
        PointStruct(
            id=i,                     # رقم بسيط مؤقت (هنستخدم نظام أفضل بعدين)
            vector=embeddings[i].tolist(),
            payload={
                "pubmed_id": row["PMID"],
                "title": row["title"],
                "content": row["content"],
            }
        )
    )

client.upsert(collection_name=COLLECTION_NAME, points=points)

print(f"Uploaded {len(points)} points.")
print("Collection info:", client.get_collection(COLLECTION_NAME))

Uploaded 5 points.
Collection info: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=5 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_un

In [12]:
# نجرب نبحث بنفس أول جملة اللي رفعناها، ونشوف هل النظام هيرجعها كأقرب نتيجة
query_text = "camomile antipeptic activity bisabolol"
query_vector = model.encode(query_text, normalize_embeddings=True).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=3
)

for point in results.points:
    print(f"Score: {point.score:.4f}")
    print(f"Title: {point.payload['title']}")
    print()

Score: 0.7705
Title: [Biochemical studies on camomile components/III. In vitro studies about the antipeptic activity of (--)-alpha-bisabolol (author's transl)].

Score: 0.4741
Title: Pharmacological properties of new neuroleptic compounds.

Score: 0.4318
Title: Influence of a new virostatic compound on the induction of enzymes in rat liver.



In [13]:
client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)
print("Clean collection ready:", client.get_collections())

Clean collection ready: collections=[CollectionDescription(name='aradoc_pubmed')]


In [14]:
from tqdm.auto import tqdm

TARGET = 100_000
BATCH_SIZE = 128

ds_full = load_dataset("MedRAG/pubmed", split="train", streaming=True)

batch_texts, batch_payloads = [], []
point_id = 0
pbar = tqdm(total=TARGET, desc="Ingesting")

for row in ds_full:
    if point_id >= TARGET:
        break

    batch_texts.append(row["contents"])
    batch_payloads.append({
        "pubmed_id": row["PMID"],
        "title": row["title"],
        "content": row["content"],
    })

    if len(batch_texts) >= BATCH_SIZE:
        vectors = model.encode(batch_texts, normalize_embeddings=True, batch_size=BATCH_SIZE, show_progress_bar=False)
        points = [
            PointStruct(id=point_id + i, vector=vectors[i].tolist(), payload=batch_payloads[i])
            for i in range(len(vectors))
        ]
        client.upsert(collection_name=COLLECTION_NAME, points=points)

        point_id += len(vectors)
        pbar.update(len(vectors))
        batch_texts, batch_payloads = [], []

# رفع أي باقي أقل من batch كامل
if batch_texts:
    vectors = model.encode(batch_texts, normalize_embeddings=True, batch_size=BATCH_SIZE, show_progress_bar=False)
    points = [
        PointStruct(id=point_id + i, vector=vectors[i].tolist(), payload=batch_payloads[i])
        for i in range(len(vectors))
    ]
    client.upsert(collection_name=COLLECTION_NAME, points=points)
    point_id += len(vectors)
    pbar.update(len(vectors))

pbar.close()
print(f"\nDone. Total points uploaded: {point_id}")

Resolving data files:   0%|          | 0/1166 [00:00<?, ?it/s]

Ingesting:   0%|          | 0/100000 [00:00<?, ?it/s]


Done. Total points uploaded: 100096


In [16]:
info = client.get_collection(COLLECTION_NAME)
print("Points count:", info.points_count)
print("Status:", info.status)

Points count: 100096
Status: green


In [17]:
!pip install -q FlagEmbedding

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.8 MB/s eta 0:00:00


In [18]:
from FlagEmbedding import FlagReranker

reranker = FlagReranker("BAAI/bge-reranker-base", use_fp16=True)
print("Reranker loaded.")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reranker loaded.
